In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from xgboost import XGBClassifier

sys.path.append(os.path.abspath(".."))

from src.data import get_xy
from src.models import evaluate_model, print_results, print_test_metrics, get_permutation_importance
from src.visalization import plot_correlation, plot_calibration, plot_weekly_xg, plot_roc_curve, plot_feature_importances

In [ ]:
df_train = pd.read_excel("training_barcelona_shots.xlsx", header=0)
df_test = pd.read_excel("testing_barcelona_shots.xlsx", header=0)

shot_features = [
    "distance_d",
    "angle_d",
    "free_kick_flag",
    "penalty_flag",
    "technique_b",
    "n_def_1_5",
    "n_def_3_0",
    "dist_nearest_def",
    "gk_dist_to_shooter"
]

X_train, y_train = get_xy(df_train, shot_features)
X_test, y_test = get_xy(df_test, shot_features)

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 1, 5]
}

In [ ]:
grid, results = evaluate_model(model, param_grid, X_train, y_train)
print_results(results)

In [ ]:
final_model_xgb = XGBClassifier(**grid.best_params_, eval_metric="logloss", random_state=42)
final_model_xgb.fit(X_train, y_train)

y_pred_proba_xgb = final_model_xgb.predict_proba(X_test)[:, 1]

In [ ]:
df_test["predicted_xg"] = y_pred_proba_xgb

statsbomb_xg = df_test["statsbomb_xg"]
pred_xg = df_test["predicted_xg"]

total_pred_xg = pred_xg.sum()
total_statsbomb_xg = statsbomb_xg.sum()
total_goals = y_test.sum()

print(f"Total Predicted xG: {total_pred_xg:.2f}")
print(f"Total StatsBomb xG: {total_statsbomb_xg:.2f}")
print(f"Actual Goals: {total_goals}")

In [ ]:
print_test_metrics(y_test, pred_xg, statsbomb_xg)
correlation = np.corrcoef(statsbomb_xg, pred_xg)[0, 1]
mae = np.mean(np.abs(statsbomb_xg - pred_xg))
print(f"\nCorrelation (Model xG vs StatsBomb xG): {correlation:.3f}")
print(f"Mean Absolute Error: {mae:.3f}")

In [ ]:
plot_correlation(statsbomb_xg, pred_xg)

In [ ]:
plot_calibration(y_test, pred_xg, statsbomb_xg)

In [ ]:
week_xg = df_test.groupby('week').agg({
    'predicted_xg': 'sum',
    'statsbomb_xg': 'sum',
    'goal/no goal': 'sum'
}).reset_index()

plot_weekly_xg(week_xg)

In [ ]:
plot_roc_curve(y_test, pred_xg, statsbomb_xg)

In [ ]:
importances_df = get_permutation_importance(final_model_xgb, X_test, y_test)
plot_feature_importances(importances_df, "XGBoost Classifier")